In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from urllib.parse import unquote



In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve
from sklearn.utils.class_weight import compute_class_weight
import joblib
 
df = pd.read_csv('waf_service/http.csv')

df['REQUEST_SIZE'] = pd.to_numeric(df['REQUEST_SIZE'], errors='coerce')
df['RESPONSE_CODE'] = pd.to_numeric(df['RESPONSE_CODE'], errors='coerce')

#1. Feature Engineering
def extract_features(df):
    """Извлечение признаков, устойчивых к evasion-атакам"""
    
    # Текстовые колонки для векторизации
    df['payload_text'] = df['MATCHED_VARIABLE_VALUE'].fillna('') + ' ' + \
                         df['MATCHED_VARIABLE_NAME'].fillna('') + ' ' + \
                         df['MATCHED_VARIABLE_SRC'].fillna('')
    
    df['req_size_log'] = np.log1p(df['REQUEST_SIZE'].fillna(0).astype(float))
    df['is_error_code'] = df['RESPONSE_CODE'].isin([403, 404, 500, 502, 503]).astype(int)
    df['ua_length'] = df['CLIENT_USERAGENT'].fillna('').str.len()
    
    ua = df['CLIENT_USERAGENT'].fillna('').str.lower()
    df['is_bot'] = ua.str.contains('bot|crawler|spider|scanner', regex=True).astype(int)
    df['is_tool'] = ua.str.contains('curl|wget|python|sqlmap|nikto|dirbuster|openvas', regex=True).astype(int)
  
    val = df['MATCHED_VARIABLE_VALUE'].fillna('').str.lower()
    
    # SQL инъекции (нормализованные)
    df['has_sql_keywords'] = val.str.contains(
        r'\b(select|union|insert|delete|update|drop|exec|declare|sleep|benchmark|waitfor)\b', 
        regex=True, case=False
    ).astype(int)
    
    df['has_sql_comments'] = val.str.contains(r'/\*|\*/|--\s|#', regex=True).astype(int)
    
    # XSS
    df['has_xss'] = val.str.contains(
        r'<script|<img|<svg|onerror|onload|javascript:|alert\(', 
        regex=True, case=False
    ).astype(int)
    
    # Path Traversal
    df['has_path_traversal'] = val.str.contains(
        r'\.\./|\.\.\\|%2e%2e|/etc/passwd|win\.ini|boot\.ini', 
        regex=True, case=False
    ).astype(int)
    
    # Command Injection
    df['has_cmd_injection'] = val.str.contains(
        r'\$\{|`|\bls\b|\bcat\b|\bpwd\b|\bid\b|;|\|', 
        regex=True, case=False
    ).astype(int)
    
    # Кодировки
    df['has_url_encoding'] = val.str.contains(r'%[0-9a-fA-F]{2}', regex=True).astype(int)
    df['has_hex_encoding'] = val.str.contains(r'0x[0-9a-fA-F]+', regex=True).astype(int)
    
    # Энтропия строки (бинарные/закодированные данные)
    df['payload_entropy'] = val.apply(lambda x: _entropy(str(x)) if len(str(x)) > 0 else 0)
    
    # Длина и спецсимволы
    df['payload_len'] = val.str.len()
    df['special_chars_ratio'] = val.str.replace(r'[a-zA-Z0-9\s]', '', regex=True).str.len() / \
                                df['payload_len'].clip(lower=1)
    
    # MATCHED_VARIABLE_SRC – где сработало правило
    df['matched_src_is_headers'] = df['MATCHED_VARIABLE_SRC'].str.contains('HEADERS', na=False).astype(int)
    df['matched_src_is_cookies'] = df['MATCHED_VARIABLE_SRC'].str.contains('COOKIES', na=False).astype(int)
    df['matched_src_is_uri'] = df['MATCHED_VARIABLE_SRC'].str.contains('URI', na=False).astype(int)
    
    return df

def _entropy(s):
    """Энтропия Шеннона для строки"""
    if len(s) <= 1:
        return 0
    prob = [s.count(c) / len(s) for c in set(s)]
    return -sum(p * np.log2(p) for p in prob)

#2. Создание целевой переменной (is_attack) ───
def label_request(row):
    """Разметка: 1 - атака, 0 - норма"""
    if pd.isna(row['MATCHED_VARIABLE_VALUE']) or str(row['MATCHED_VARIABLE_VALUE']).strip() == '':
        return 0
    
    value = str(row['MATCHED_VARIABLE_VALUE']).lower()
    user_agent = str(row.get('CLIENT_USERAGENT', '')).lower()
    
    # Поисковые боты = норма
    if any(bot in user_agent for bot in ['yandex', 'googlebot', 'bingbot', 'ahrefs', 'sputnik']):
        return 0
    
    # Явные признаки атак
    if re.search(r'(union.*select|select.*from|insert.*into)', value, re.IGNORECASE):
        return 1
    if re.search(r'/\*.*\*/|--\s', value):
        return 1
    if re.search(r'<script|<img.*onerror|javascript:|alert\(', value, re.IGNORECASE):
        return 1
    if re.search(r'\.\./.*(?:etc/passwd|win\.ini|boot\.ini)', value):
        return 1
    if re.search(r'\$\{|`.*`|\bls\s+\-la\b', value):
        return 1
    if re.search(r"(?:\d+|')\s*(?:or|and)\s*(?:\d+|')\s*=", value):
        return 1
    
    return 0

df['is_attack'] = df.apply(label_request, axis=1)
print("Распределение классов:")
print(df['is_attack'].value_counts())
print(f"Доля атак: {df['is_attack'].mean():.2%}")

# 3. Подготовка данных
df_featured = extract_features(df.copy())

# Заполняем пропуски в признаках
df_featured['payload_text'] = df_featured['payload_text'].fillna('')

# Определяем признаки и целевую переменную
text_features = ['payload_text']
numeric_features = [
    'req_size_log', 'ua_length', 'payload_len', 
    'payload_entropy', 'special_chars_ratio'
]
binary_features = [
    'is_error_code', 'is_bot', 'is_tool',
    'has_sql_keywords', 'has_sql_comments', 'has_xss',
    'has_path_traversal', 'has_cmd_injection',
    'has_url_encoding', 'has_hex_encoding',
    'matched_src_is_headers', 'matched_src_is_cookies', 'matched_src_is_uri'
]

# Проверяем что все колонки существуют
all_features = text_features + numeric_features + binary_features
missing = [f for f in all_features if f not in df_featured.columns]
if missing:
    print(f"Отсутствующие колонки: {missing}")

X = df_featured[all_features].copy()
y = df_featured['is_attack']

# Проверка на пропуски
print(f"\nПропуски в X: {X.isna().sum().sum()}")
X = X.fillna(0)  # Заполняем оставшиеся пропуски нулями

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print(f"Train: {X_train.shape}, Атак: {y_train.sum()} ({y_train.mean():.2%})")
print(f"Test:  {X_test.shape}, Атак: {y_test.sum()} ({y_test.mean():.2%})")

#4. Построение пайплайна
preprocessor = ColumnTransformer([
    ('text', TfidfVectorizer(max_features=1000, ngram_range=(1,3)), 'payload_text'),
    ('numeric', StandardScaler(), numeric_features),
    ('binary', 'passthrough', binary_features)
], remainder='drop')

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(
        n_estimators=200,
        max_depth=5,
        learning_rate=0.05,
        min_samples_leaf=20,
        random_state=42
    ))
])

#5. Обучение с учетом дисбаланса классов ───
class_weights = compute_class_weight(
    'balanced', classes=np.array([0, 1]), y=y_train
)
sample_weights = np.where(y_train == 1, class_weights[1], class_weights[0])

pipeline.fit(X_train, y_train, classifier__sample_weight=sample_weights)

# 6. Оценка модели ───
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]

print("\n" + "="*60)
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['Normal', 'Attack']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"True Negatives: {cm[0,0]}, False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}, True Positives: {cm[1,1]}")

print(f"\nROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")

# Кросс-валидация
cv_scores = cross_val_score(
    pipeline, X_train, y_train, 
    cv=StratifiedKFold(5, shuffle=True, random_state=42),
    scoring='roc_auc'
)
print(f"CV ROC-AUC (mean ± std): {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

#7. Подбор оптимального порога ───
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

precision, recall, thresholds = precision_recall_curve(y_test, y_pred_proba)

# Визуализация Precision-Recall кривой
plt.figure(figsize=(10, 6))
plt.plot(thresholds, precision[:-1], label='Precision', color='blue')
plt.plot(thresholds, recall[:-1], label='Recall', color='red')
plt.axvline(x=0.05, color='green', linestyle='--', label='Threshold=0.05')
plt.axvline(x=0.1, color='orange', linestyle='--', label='Threshold=0.1')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.title('Precision-Recall vs Threshold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.savefig('precision_recall_curve.png', dpi=100, bbox_inches='tight')
plt.show()

# Показываем варианты порогов
print("\nВарианты порогов:")
print("-" * 60)
print(f"{'Target Recall':<15} {'Threshold':<12} {'Precision':<12} {'Actual Recall':<12}")
print("-" * 60)

best_threshold = 0.05  # начальное значение
best_f1 = 0

for target_recall in [0.85, 0.88, 0.90, 0.92, 0.95, 0.97, 0.99]:
    idx = np.argmax(recall >= target_recall)
    if idx < len(thresholds):
        thresh = thresholds[idx]
        prec = precision[idx]
        actual_recall = recall[idx]
        
        # Вычисляем F1-score для этого порога
        y_pred_temp = (y_pred_proba >= thresh).astype(int)
        from sklearn.metrics import f1_score
        f1 = f1_score(y_test, y_pred_temp)
        
        print(f"{target_recall:<15} {thresh:<12.6f} {prec:<12.4f} {actual_recall:<12.4f}")
        
        # Выбираем порог с лучшим F1-score
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = thresh


# Выбираем порог: компромисс между precision и recall
# Для production: важно не пропускать атаки, но и не блокировать легитимных пользователей
target_recall = 0.92 
idx = np.argmax(recall >= target_recall)
optimal_threshold = thresholds[idx] if idx < len(thresholds) else 0.1

# Если порог всё ещё слишком низкий (< 0.01), используем фиксированный
if optimal_threshold < 0.01:
    optimal_threshold = 0.05

# Финальный тест
y_pred_optimal = (y_pred_proba >= optimal_threshold).astype(int)
print(f"\nРезультаты с порогом {optimal_threshold:.6f}:")
print(classification_report(y_test, y_pred_optimal, target_names=['Normal', 'Attack']))

# Проверяем на конкретных примерах
print("\nТест на примерах:")
test_values = [
    ("page=home", "Mozilla/5.0 Chrome"),           # должно быть Normal
    ("1' UNION SELECT * FROM users--", "sqlmap"),   # должно быть Attack
    ("<script>alert(1)</script>", "Mozilla/5.0"),   # должно быть Attack
    ("contact", "Mozilla/5.0"),                      # должно быть Normal
]

for value, ua in test_values:
    test_df = pd.DataFrame([{
        'payload_text': value,
        'req_size_log': 5.0,
        'ua_length': len(ua),
        'payload_len': len(value),
        'payload_entropy': 2.0,
        'special_chars_ratio': 0.1,
        'is_error_code': 0,
        'is_bot': 0,
        'is_tool': 1 if ua == 'sqlmap' else 0,
        'has_sql_keywords': 1 if 'select' in value.lower() else 0,
        'has_sql_comments': 1 if '--' in value else 0,
        'has_xss': 1 if '<script>' in value else 0,
        'has_path_traversal': 0,
        'has_cmd_injection': 0,
        'has_url_encoding': 0,
        'has_hex_encoding': 0,
        'matched_src_is_headers': 0,
        'matched_src_is_cookies': 0,
        'matched_src_is_uri': 0
    }])
    
    proba = pipeline.predict_proba(test_df)[0, 1]
    pred = 1 if proba >= optimal_threshold else 0
    print(f"  '{value[:50]:<50}' -> {'АТАКА' if pred else 'НОРМА'} (prob={proba:.4f})")

# ─── 8. Сохранение модели ───
import os
os.makedirs('models', exist_ok=True)

joblib.dump(pipeline, 'models/waf_model.pkl')
joblib.dump(optimal_threshold, 'models/threshold.pkl')

print("Модель сохранена в models/waf_model.pkl")
print("Порог сохранён в models/threshold.pkl")

In [ ]:
import os
import joblib

os.makedirs('models', exist_ok=True)

# Сохраняем pipeline и порог
joblib.dump(pipeline, 'models/waf_model.pkl')
joblib.dump(optimal_threshold, 'models/threshold.pkl')

import json
feature_info = {
    'text_features': ['payload_text'],
    'numeric_features': ['req_size_log', 'ua_length', 'payload_len', 
                         'payload_entropy', 'special_chars_ratio'],
    'binary_features': ['is_error_code', 'is_bot', 'is_tool',
                        'has_sql_keywords', 'has_sql_comments', 'has_xss',
                        'has_path_traversal', 'has_cmd_injection',
                        'has_url_encoding', 'has_hex_encoding',
                        'matched_src_is_headers', 'matched_src_is_cookies', 'matched_src_is_uri']
}
with open('models/feature_info.json', 'w') as f:
    json.dump(feature_info, f)